# 00 - Prepare Features from OTEL Datasets

**Elyra Pipeline Node 0** — Extract features from raw OTEL datasets (logs, traces, metrics). Runs inside the pipeline so no external `feature_pipeline.py` is needed on RHOAI.

**Inputs** (from pipeline or env):
- `OTEL_DATASET_DIR`: Path to datasets from git clone (default: `../datasets` = `performance_comparison/datasets`)

**Outputs**:
- `OUT_FEATURES_CSV`: features.csv for train/test notebooks

In [ ]:
import os
import json
import re
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Set

import pandas as pd

# Datasets from git clone: ../datasets when run from elyra_pipeline; or set REPO_ROOT for absolute path
REPO_ROOT = os.getenv("REPO_ROOT", "")
DEFAULT_DATASETS = Path(REPO_ROOT) / "research/anomaly_detection/performance_comparison/datasets" if REPO_ROOT else Path("../datasets")
OTEL_DATASET_DIR = Path(os.getenv("OTEL_DATASET_DIR", str(DEFAULT_DATASETS)))
OUT_FEATURES_CSV = Path(os.getenv("OUT_FEATURES_CSV", "artifacts/features.csv"))

OUT_FEATURES_CSV.parent.mkdir(parents=True, exist_ok=True)
print(f"Dataset dir: {OTEL_DATASET_DIR}")
print(f"Output: {OUT_FEATURES_CSV}")

In [ ]:
@dataclass
class Sample:
    sample_id: str
    label: str
    is_baseline: bool
    root_cause: str
    flag: str
    variant: str
    paths: Dict[str, Path]


def discover_samples(data_dir: Path) -> List[Sample]:
    samples = []
    patterns = ["metadata_exp*.json", "metadata_val*.json", "metadata_test*.json",
                "metadata_eval*.json", "metadata_*baseline*.json", "metadata_train*.json"]
    metas = set()
    for pat in patterns:
        for p in data_dir.rglob(pat):
            metas.add(p.resolve())
    for meta_path in sorted(metas):
        try:
            with open(meta_path) as f:
                meta = json.load(f)
        except Exception:
            continue
        label = meta.get("label") or meta_path.stem.replace("metadata_", "")
        root_cause = meta.get("ground_truth_root_cause", "unknown")
        flag = meta.get("flag", "")
        variant = meta.get("variant", "")
        base_dir = meta_path.parent
        paths = {"metadata": meta_path, "logs": base_dir / f"logs_{label}.txt",
                 "traces": base_dir / f"traces_{label}.json", "metrics": base_dir / f"metrics_{label}.json"}
        samples.append(Sample(label, label, root_cause == "none", root_cause, flag, variant, paths))
    return samples


def count_log_errors(log_path: Path) -> int:
    if not log_path.exists():
        return 0
    err = re.compile(r"(?is)(error[: ]|\berror\b|\bexception\b|\bfail(?:ed|ure)?\b|\bunavailable\b|level=error)")
    infra = re.compile(r"^\s*(otel-collector|grafana|jaeger|prometheus|opensearch)\b", re.I)
    n = 0
    with open(log_path, "r", errors="ignore") as f:
        for line in f:
            if infra.search(line):
                continue
            if err.search(line):
                n += 1
    return n


def count_traces(tp: Path) -> int:
    if not tp.exists():
        return 0
    try:
        with open(tp) as f:
            d = json.load(f)
        if isinstance(d, list):
            return len(d)
        if isinstance(d, dict) and "hits" in d and "hits" in d["hits"]:
            return len(d["hits"]["hits"])
    except Exception:
        pass
    return 0


def trace_stats(tp: Path) -> Dict[str, int]:
    st = {"trace_total_spans": 0, "trace_http_5xx_spans": 0, "trace_unique_services": 0, "trace_unique_span_names": 0}
    if not tp.exists():
        return st
    try:
        with open(tp) as f:
            d = json.load(f)
        if isinstance(d, dict) and "hits" in d and isinstance(d["hits"], dict) and "hits" in d["hits"]:
            hits = d["hits"]["hits"]
        elif isinstance(d, list):
            hits = d
        else:
            hits = []
        svc, names = set(), set()
        for h in hits:
            src = h.get("_source", {}) if isinstance(h, dict) else {}
            st["trace_total_spans"] += 1
            attrs = src.get("attributes", {}) or {}
            if int(attrs.get("http.status_code", -1)) >= 500:
                st["trace_http_5xx_spans"] += 1
            res = src.get("resource", {}) or {}
            rattr = res.get("attributes", {}) or {}
            s = rattr.get("service.name") or attrs.get("service.name", "")
            if s:
                svc.add(s)
            if src.get("name") or src.get("spanName"):
                names.add(str(src.get("name") or src.get("spanName")))
        st["trace_unique_services"] = len(svc)
        st["trace_unique_span_names"] = len(names)
    except Exception:
        pass
    return st


def count_log_lines(lp: Path) -> int:
    return sum(1 for _ in open(lp, "r", errors="ignore")) if lp.exists() else 0


def count_service_errors(lp: Path, prefix: str) -> int:
    if not lp.exists():
        return 0
    err = re.compile(r"(?is)(error[: ]|\\berror\\b|\\bexception\\b|level=error)")
    svc = re.compile(rf"^\s*{re.escape(prefix)}\b", re.I)
    n = 0
    with open(lp, "r", errors="ignore") as f:
        for line in f:
            if svc.search(line) and err.search(line):
                n += 1
    return n


def count_metric_series(mp: Path) -> int:
    if not mp.exists():
        return 0
    try:
        d = json.load(open(mp))
        return sum(len(v) for v in d.values() if isinstance(v, list)) if isinstance(d, dict) else 0
    except Exception:
        return 0


def metrics_scalar_sum(mp: Path, key: str) -> float:
    if not mp.exists():
        return 0.0
    try:
        d = json.load(open(mp))
        total = 0.0
        for e in (d.get(key) or []):
            v = e.get("value") if isinstance(e, dict) else None
            if isinstance(v, list) and len(v) >= 2:
                total += float(v[1])
        return total
    except Exception:
        return 0.0


def build_features(samples: List[Sample]) -> pd.DataFrame:
    rows = []
    for s in samples:
        tc = count_traces(s.paths["traces"])
        ts = trace_stats(s.paths["traces"])
        lec = count_log_errors(s.paths["logs"])
        tll = count_log_lines(s.paths["logs"])
        ler = (lec / tll) if tll else 0.0
        lb = s.label.lower()
        in_tr = s.is_baseline and (lb.startswith("train") or "_train" in lb)
        rows.append({"id": s.sample_id, "label": s.label, "flag": s.flag, "variant": s.variant, "root_cause": s.root_cause,
                    "is_baseline": s.is_baseline, "in_training": in_tr, "trace_count": tc, **ts,
                    "log_error_count": lec, "log_total_lines": tll, "log_error_ratio": ler,
                    "frontend_error_count": count_service_errors(s.paths["logs"], "frontend"),
                    "checkout_error_count": count_service_errors(s.paths["logs"], "checkout"),
                    "recommendation_error_count": count_service_errors(s.paths["logs"], "recommendation"),
                    "product_catalog_error_count": count_service_errors(s.paths["logs"], "product-catalog"),
                    "payment_error_count": count_service_errors(s.paths["logs"], "payment"),
                    "metrics_series_len": count_metric_series(s.paths["metrics"]),
                    "req_rate_sum": metrics_scalar_sum(s.paths["metrics"], "req_rate")})
    return pd.DataFrame(rows)

print("Feature extraction helpers defined.")

In [ ]:
import numpy as np

if OTEL_DATASET_DIR.exists() and list(OTEL_DATASET_DIR.rglob("metadata_*.json")):
    samples = discover_samples(OTEL_DATASET_DIR)
    if samples:
        df = build_features(samples)
        df.to_csv(OUT_FEATURES_CSV, index=False)
        n_train = int(df["in_training"].sum())
        print(f"Built features from {len(samples)} samples → {OUT_FEATURES_CSV}")
        print(f"  Train rows: {n_train}, Test rows: {len(df) - n_train}")
    else:
        raise ValueError(f"No samples discovered in {OTEL_DATASET_DIR}")
else:
    # No datasets: generate synthetic features for pipeline demo
    print("No datasets found. Generating synthetic features...")
    np.random.seed(42)
    cols = ["trace_count", "trace_total_spans", "trace_http_5xx_spans", "log_error_count", "log_error_ratio",
            "metrics_series_len", "req_rate_sum", "trace_unique_services", "trace_unique_span_names",
            "frontend_error_count", "checkout_error_count", "recommendation_error_count",
            "product_catalog_error_count", "payment_error_count", "log_total_lines"]
    n_train, n_eval = 30, 80
    X_train = np.random.exponential(5, (n_train, len(cols))).clip(0, 50)
    X_eval_n = np.random.exponential(5, (40, len(cols))).clip(0, 50)
    X_eval_a = np.random.exponential(20, (40, len(cols))).clip(10, 200)
    X = np.vstack([X_train, X_eval_n, X_eval_a])
    df = pd.DataFrame(X, columns=cols)
    df["id"] = [f"train{i:02d}" for i in range(n_train)] + [f"eval{i:02d}" for i in range(80)]
    df["label"] = df["id"]
    df["flag"] = ["baseline"] * n_train + ["baseline"] * 40 + ["fault"] * 40
    df["variant"] = "off"
    df["root_cause"] = ["none"] * (n_train + 40) + ["fault"] * 40
    df["is_baseline"] = df["root_cause"] == "none"
    df["in_training"] = [True] * n_train + [False] * 80
    df.to_csv(OUT_FEATURES_CSV, index=False)
    print(f"Saved synthetic features → {OUT_FEATURES_CSV}")